# Rolling plot of ensemble


In [ ]:
import openmeteo_requests

import pandas as pd
import matplotlib.pyplot as plt
import requests_cache
from retry_requests import retry

In [ ]:
import earthkit.data as ekd
import xarray as xr

In [ ]:
station = "Uccle"
latitude = 50.797
longitude = 4.357

## Fetching observation from the Open Meteo website

In [ ]:
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
arch_url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 50.797,  # location of the RMI Uccle weather station (change if you want)
	"longitude": 4.357,
    "start_date": "2026-05-01",
	"end_date": "2026-07-27",
	# "hourly": ["temperature_2m"],
    "daily": "temperature_2m_max",
    # "models": "era5",
}
arch_responses = openmeteo.weather_api(arch_url, params = params)
arch_response = arch_responses[0]

In [ ]:
# code for daily data

# arch_hourly = arch_response.Hourly()
# arch_hourly_temperature_2m = arch_hourly.Variables(0).ValuesAsNumpy()

# arch_hourly_data = {
# 	"date": pd.date_range(
# 		start = pd.to_datetime(arch_hourly.Time(), unit="s", utc=True),
# 		end =  pd.to_datetime(arch_hourly.TimeEnd(), unit="s", utc=True),
# 		freq = pd.Timedelta(arch_hourly.Interval(), unit = "s"),
# 		inclusive = "left"
# 	)
# }

# arch_hourly_data["temperature_2m_obs"] = arch_hourly_temperature_2m

# arch_hourly_dataframe = pd.DataFrame(data = arch_hourly_data).set_index("date")
# observations = arch_hourly_dataframe

In [ ]:
arch_daily = arch_response.Daily()
arch_daily_temperature_2m_max = arch_daily.Variables(0).ValuesAsNumpy()

arch_daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(arch_daily.Time(), unit="s", utc=True),
		end =  pd.to_datetime(arch_daily.TimeEnd(), unit="s", utc=True),
		freq = pd.Timedelta(arch_daily.Interval(), unit = "s"),
		inclusive = "left"
	)
}

arch_daily_data["temperature_2m_max_obs"] = arch_daily_temperature_2m_max

arch_daily_dataframe = pd.DataFrame(data = arch_daily_data).set_index("date")
observations = arch_daily_dataframe

In [ ]:
# quick plot to check
observations.plot()

## Fetching ensemble forecasts with earthkit

You need an ECMWF account to fetch these data. If you don't have one, it will ask to create one.

In [ ]:
for i in range(len(observations.index)):

    # determining the date to process
    isodate = str(observations.index[i].year)+'-'+str(observations.index[i].month).rjust(2, "0")+'-'+str(observations.index[i].day).rjust(2, "0")
    
    # we now fetch an advanced request to get the ensemble
    request = {
        "class":   "od",          # Operational Data (IFS)
        "stream":  "enfo",        # Ensemble forecast system (atmospheric fields)
        "type":    "pf",          # Perturbed Forecast members
        "date":    isodate,  # Forecast initialization date "YYYY-MM-DD"
        "time":    "00",          # Model run base time (00 or 12)
        # "step":    model_steps,   # Forecast step in hours (e.g., 12 hours from base time)
        "step":    "all",   # Forecast step in hours (e.g., 12 hours from base time)
        "levtype": "sfc",         # Surface level variables
        "number": "1/to/50",      # Retrieve all 50 perturbed members
        "param":   ["mx2t6"],  # We ask for 2 metre temperature, mean sea level pressure, and total precipitation (hourly accumlated)
        "area":    [52, 3.5, 49, 6],    # N,W,S,E - domain requested in latitude - longitude coordinates
        # "grid":    [0.25, 0.25],         # We ask for a latitude-longitude grid with a point every 0.25°
    }
    
    source = ekd.from_source(
        "mars",
        **request
    )
    
    # converting to xarray dataset
    ds = source.to_xarray(time_dims=["valid_time"])

    # extraction daily T max
    dsm = ds.resample(valid_time="1D").max()
    
    # creating an index for the latitude and longitude
    dss = dsm.set_xindex(("latitude", "longitude"), xr.indexes.NDPointIndex)
    
    # selecting the station point
    sel_station = dss.sel(latitude=latitude, longitude=longitude, method="nearest")

    # saving to netcdf
    sel_station.to_netcdf(f'IFS_ENS_{station}_{isodate}.nc')

In [ ]:
# quick plot to test
plt.figure(figsize=(10,5))
ax = plt.gca()
for i in range(len(sel_station.member)):
    (sel_station['mx2t6'].isel(member=i) - 273.15).plot(ax=ax, color='tab:purple')

observations.plot(ax=ax, color='tab:green')
plt.xlabel('date')
plt.ylabel('Temperature at 2 metre [°C]')